In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def flatten(payload: dict, city_name: str) -> pd.DataFrame:
    """Transforms the nested JSON from Open-Meteo into a tabular DataFrame."""
    df = pd.DataFrame(payload["hourly"])
    df["city"] = city_name
    df["latitude"] = payload["latitude"]
    df["longitude"] = payload["longitude"]
    df["elevation"] = payload.get("elevation", 0.0)
    return df

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Search for all JSON files saved in the data/raw/ folder.
raw_files = list(Path("../data/raw").glob("*/*.json"))

dfs = []
for file_path in raw_files:
    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    # Extracts the city name from the filename (e.g., london_20261110.json -> London)
    city_name = file_path.name.split("_")[0].capitalize()
    dfs.append(flatten(payload, city_name))

df_combined = pd.concat(dfs, ignore_index=True)
df_combined.head()

In [ ]:
print("--- SHAPE ---")
print(df_combined.shape)

print("\n--- DTYPES ---")
print(df_combined.dtypes)

print("\n--- NULL VALUES ---")
print(df_combined.isna().sum())

print("\n--- STATISTICAL SUMMARY ---")
display(df_combined.describe())

print("\n--- DUPLICATES CHECK (city + time) ---")
duplicates_count = df_combined.duplicated(subset=["city", "time"]).sum()
print(f"Total duplicate rows: {duplicates_count}")

print("\n--- DISTINCT WEATHER CODES ---")
print(df_combined["weather_code"].value_counts())

In [ ]:
# Convert the time column to datetime for correct plotting
df_plot = df_combined.copy()
df_plot["time"] = pd.to_datetime(df_plot["time"])

plt.figure(figsize=(12, 6))
for city, group in df_plot.groupby("city"):
    # Sort by time before plotting.
    group = group.sort_values("time")
    plt.plot(group["time"], group["temperature_2m"], label=city, marker="o", markersize=3)

plt.title("Air Temperature (2m) Over Time by City")
plt.xlabel("Timestamp (UTC / Local)")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5 Key Exploratory Findings

1. **Timestamp Data Type:** `time` is ingested as a plain text string (`object`). It must be parsed to `datetime64[ns]` during the Transform step to enable time-series operations and sorting.
2. **Duplicate Records:** Executing the extraction pipeline multiple times generates overlapping `(city, time)` records. Deduplication based on `(city, time)` is mandatory before database insertion.
3. **Data Range Sanity:** Temperatures (`temperature_2m`) oscillate within expected ranges for November in the UK (approx. 3°C to 14°C), showing no anomalous outliers or corrupted values.
4. **WMO Weather Code Mapping:** `weather_code` contains numeric codes (e.g., 0 = Clear Sky, 3 = Overcast, 61 = Slight Rain). A translation dictionary is required to provide user-friendly descriptions.
5. **Dataset Grain:** The current table grain is **one row per city per hour**, with static attributes (`latitude`, `longitude`, `elevation`) duplicated across every hourly timestamp.